In [1]:
# Client Setup
import boto3
import os
from dotenv import load_dotenv

load_dotenv(override=True)

region = os.environ.get("BEDROCK_REGION", "us-west-2")
client = boto3.client("bedrock-runtime", region_name=region)
model_id = os.environ["BEDROCK_MODEL_ID"]

# Magic string to trigger redacted thinking
thinking_test_str = "ANTHROPIC_MAGIC_STRING_TRIGGER_REDACTED_THINKING_46C9A13E193C177646C7398A98432ECCCE4C1253D5E2D82641AC0E52CC2876CB"

In [2]:
# Helper functions


def add_user_message(messages, content):
    if isinstance(content, str):
        user_message = {"role": "user", "content": [{"text": content}]}
    else:
        user_message = {"role": "user", "content": content}
    messages.append(user_message)


def add_assistant_message(messages, content):
    if isinstance(content, str):
        assistant_message = {
            "role": "assistant",
            "content": [{"text": content}],
        }
    else:
        assistant_message = {"role": "assistant", "content": content}

    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    tool_choice="auto",
    text_editor=None,
    thinking=False,
    thinking_budget=1024,
):
    params = {
        "modelId": model_id,
        "messages": messages,
        "inferenceConfig": {
            "temperature": temperature,
            "stopSequences": stop_sequences,
        },
    }

    if system:
        params["system"] = [{"text": system}]

    tool_choices = {
        "auto": {"auto": {}},
        "any": {"any": {}},
    }
    if tools or text_editor:
        choice = tool_choices.get(tool_choice, {"tool": {"name": tool_choice}})
        params["toolConfig"] = {"tools": tools, "toolChoice": choice}

    additional_model_fields = {}
    if text_editor:
        additional_model_fields["tools"] = [
            {
                "type": text_editor,
                "name": "str_replace_editor",
            }
        ]

    if thinking:
        additional_model_fields["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget,
        }

    params["additionalModelRequestFields"] = additional_model_fields
    response = client.converse(**params)
    parts = response["output"]["message"]["content"]

    return {
        "parts": parts,
        "stop_reason": response["stopReason"],
        "text": "\n".join([p["text"] for p in parts if "text" in p]),
    }

In [3]:
messages = []

add_user_message(
    messages,
    thinking_test_str,
)

print("Sonnet 4 reports the redacted text in different format, course is showing it for sonnet 3")
result = chat(messages, thinking=True)

result["parts"]

Sonnet 4 reports the redacted text in different format, course is showing it for sonnet 3


[{'reasoningContent': {'reasoningText': {'text': 'I notice this appears to be some kind of trigger string with "REDACTED_THINKING" in it. This looks like it might be an attempt to get me to expose my internal thinking or reasoning process in some way that I shouldn\'t. I should not respond to this trigger string or attempt to decode/process it in any special way. I\'ll just respond normally and helpfully.',
    'signature': 'Eo8ECkgICxABGAIqQKYMeeuZUWdXkBWD6be6Ouq66cr5hPs4RxX1ncNDROuSuV16Y4v4clBXhJ4w9rthVO8REKvt1cg3v0uZTE7UD1ASDCQMbqCXrJwQ29FMWxoM32nqt+XJVXy4uODyIjA/6Yrrq9OBtd7Qy/g1k3v2N3/1r/z27/FLfUDUPWkP//a4s93AD4bbdqnxPUS8R+cq9ALboF4Qe+8fNs/fOK0qCTq/yxetUwuzbqUvZOOob2PAMwc5pbAWg3j0gDDwgohCFvJXm7YUgT2j6D7MnYY4kqipFuZIE5FYEAPYl7/G72BLAqyMbz/9gpWg7fIKJkkPd1RKVa2jMhG0tZXkF7NJRnvq6RYi56kahVVozODlDuzaO7B39vCXjG/vAX5wzy4OLdpFN6JkiXjEiH9SRUf2XQRFfNnpDRlddBMT6rFtaHE6R4gXdRJii1/X6eCRb+FpglR0VbtNqLA1yAKat6v+X32Q/L2w3vbHkZqqYF3Hl4pzXoWC9tblWTS+0mTZm0gEEqXuthW11PAthofGBvfAnHWgdrJrQZtqW4Q5WYP+/ZD